# Avance 2: ingeniería de características para estimar la emisión de NOₓ

Equipo 39 · Proyecto `nox_ml` (Ciudad de México, noviembre–diciembre de 2023)

## 1. Objetivo y reglas del ejercicio

El Avance 1 mostró cómo son los datos y qué se corrige. Este avance responde tres preguntas con **evidencia de los mismos datos**:

1. ¿Qué características se construyen, y por qué cada una tiene sentido físico? (secciones 2 y 3)
2. ¿Qué escalas y transformaciones necesita cada modelo, y cuáles no? (sección 4)
3. ¿Cuáles conservar, descartar o combinar? (sección 5)

Reglas: todo lo que decide este cuaderno sale de **noviembre–diciembre de 2023** (61 días, 366 horas, 7,537 filas estación-hora de las tardes, 12–17 h locales); febrero–mayo de 2024 no se carga ni se usa aquí, porque es el periodo de prueba del reporte final. El código solo llama a `nox_ml` (`nox_ml.validation.report_eda` y `load_inputs`): las características son las mismas que calculan los modelos (`nox_ml.models.rf_deweather.hour_table`), no una copia.

Dos clases de modelo piden cosas distintas a los datos, y por eso el cuaderno separa lo que es general de lo que es de un modelo:

- **E6, bosque aleatorio** (`rf_deweather`): aprende NOₓ ~ meteorología + calendario. Usa las características de las secciones 2 y 3 y un objetivo transformado (sección 4.1). Los árboles no necesitan escalar las entradas.
- **Inversión bayesiana y PINN** (E1–E5): trabajan con física, no con características aprendidas: el viento y la capa límite entran por el transporte. La PINN sí necesita una ecuación **adimensional** (sección 4.2).


In [ ]:
%matplotlib inline
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import report_eda as eda

DATA = eda.find_data()
inputs = load_inputs(DATA, *eda.MODEL_PERIOD)
rows = eda.feature_rows(inputs)
hours = eda.hourly_table(inputs)
start, end = eda.MODEL_PERIOD
print(f"Periodo de modelado: {start} a {end}; {len(hours)} horas, {len(rows):,} filas estación-hora")
hours[[*eda.FEATURES, "day_type"]].head()

Las columnas son las características de E6: `hour_sin/cos`, `doy_sin/cos` (hora y día del año como círculos), `wind_sin/cos` (dirección del viento), `ventilation` y sus sumas de 3 y 6 horas, más el tipo de día (0 hábil, 1 sábado, 2 domingo o festivo).

## 2. Construcción de características con justificación de dominio

| Característica | Cómo se construye | Por qué (dominio) |
|---|---|---|
| `ventilation` | velocidad del viento (m/s) × altura de la capa límite (m), en m²/s | Con emisión constante la concentración es proporcional a 1 / ventilación: es la tasa a la que el aire sobre la ciudad se renueva. Resume dos variables en la que importa |
| `ventilation_3h`, `ventilation_6h` | suma de la ventilación de las 3 y 6 horas anteriores (las horas ausentes se rellenan con la media de las presentes, y con la hora actual si no hay ninguna) | El NOₓ vive horas: lo que se acumuló con aire estancado sigue ahí. Las tardes dejan huecos en la serie, por eso el relleno |
| `hour_sin`, `hour_cos` | hora local como punto de un círculo de 24 h | Las 23 h y las 0 h son vecinas; el tráfico y la mezcla son cíclicos |
| `doy_sin`, `doy_cos` | día del año como punto de un círculo de 365.25 días | Deriva estacional de emisiones y fondo sin una tendencia monótona que no se pueda extrapolar |
| `wind_sin`, `wind_cos` | componentes del vector unitario del viento (hacia dónde sopla); (0, 0) si no hay viento | La dirección dice de qué lado de la ciudad viene el aire, y 359° y 1° son casi el mismo viento. Es lo que distingue una estación de otra |
| `day_type` | 0 día hábil, 1 sábado, 2 domingo o festivo (calendario oficial) | Es la cantidad que se quiere medir: el cambio de emisión por tipo de día. Se mantiene tal como se observó |
| proxy vial (OSM) | suma de longitud de calles ponderada por clase (autopista y troncal 3, primaria 2, secundaria 1.5, terciaria 1) en celdas de 1 km, escalada al inventario de 2018 | Dice cuánta emisión vehicular hay *cerca* de cada estación: el nivel del sitio |

### 2.1 Ventilación: ¿aporta más que sus dos factores?

Correlación de Spearman con el NOₓ de cada estación-hora:

In [ ]:
eda.ventilation_gain(rows).round(2)

La capa límite sola correlaciona −0.41 con el NOₓ y el viento sólo −0.12; la ventilación (el producto) queda en −0.34, entre las dos. **El producto no gana a la capa límite en correlación simple**, y eso hay que decirlo: lo que aporta es la forma física (concentración ∝ 1 / ventilación), con la que un árbol o una regresión pueden separar emisión de dilución, no un coeficiente más alto. Las sumas de 3 y 6 horas pierden una centésima de correlación (−0.32) pero suman información del pasado. Relación con el NOₓ por decil y por hora:

In [ ]:
eda.plot_ventilation(rows)

### 2.2 Hora del día: ¿por qué un círculo?

Distancia entre una hora y la siguiente, como número y como punto del círculo:

In [ ]:
eda.cyclic_check(24).round(3).head(3), eda.cyclic_check(24).round(3).tail(2)

Como número, el paso de las 23 h a las 0 h mide 23 y los demás 1: un modelo que lo lea así ve un salto que no existe. Como punto del círculo todos los pasos miden 0.261. **Ojo:** en estos datos solo hay horas locales de 11 a 16 (seis valores), así que `hour_sin` y `hour_cos` recorren un arco corto del círculo (sin entre −0.87 y 0.26, cos entre −1 y −0.5) y están correlacionadas (−0.83): hoy el círculo no es necesario, es el diseño correcto si algún día se amplía la ventana a la noche, y cuesta una columna.

### 2.3 Dirección del viento: seno y coseno

La dirección es un ángulo (0° = 360°); como número tiene un salto en el norte. Se usan los dos componentes del vector unitario:

In [ ]:
eda.plot_wind_rose(hours)

En las tardes de noviembre–diciembre el viento viene sobre todo del norte y el noreste (0° a 45°), con un grupo menor del sur: la dirección dominante es estable, de acuerdo con el control de calidad de SIMAT (dirección media de ~51° en la temporada seca). La nube de puntos de la derecha muestra el vector (hacia dónde sopla): se agrupa en el sur del círculo, y el círculo no tiene el salto de 359° a 1°. Con viento calmo (velocidad cero) los dos componentes valen 0, para no inventar una dirección.

### 2.4 Tipo de día

Tipo de día con sus días y los cocientes crudos (no son cocientes de emisión):

In [ ]:
eda.day_type_table(rows).round(2)

Se codifica como **tres indicadoras** (una por tipo) y no como 0/1/2: el sábado no está "entre" el día hábil y el domingo en ningún sentido ordinal que un modelo deba asumir. Los festivos (20 de noviembre, 25 de diciembre) se cuentan como domingo porque el tráfico se comporta como domingo. Quedan 41, 9 y 11 días: pocos, y por eso los intervalos de los estimadores son anchos.

### 2.5 Proxy de emisión vial de OSM

El proxy suma, para cada estación, el peso vial de las celdas de 1 km dentro de un radio. Se prueba si **ordena a las estaciones** como las ordena su nivel de NOₓ (media de log(NOₓ − fondo) en las tardes), a varios radios:

In [ ]:
eda.proxy_relation(inputs.zones, inputs.station_xy, rows).round(3)

Con 24 estaciones, el proxy ordena bien a las estaciones a escala de barrio y ciudad: el Spearman sube de 0.49 (500 m) a **0.68 a 2 km, 0.71 a 3 km y 0.74 a 5 km** (p < 0.001 en los tres), con un valle a 1 km (0.35, p = 0.09) donde el posicionamiento de cada sensor respecto de una calle concreta importa más que la red. Mapa y relación a 3 km:

In [ ]:
eda.plot_road_proxy(inputs.zones, inputs.station_xy, rows, 3000)

Dos advertencias. (1) Un Spearman de 0.7 con 24 puntos separa bien el centro denso de la periferia (Merced y Tlalnepantla arriba; Montecillo, Acolman y Villa de las Flores abajo), pero **no** explica las diferencias finas entre estaciones parecidas ni un caso como Ajusco Medio (proxy bajo y NOₓ intermedio, un sitio de montaña al sur que recibe aire de la ciudad). (2) A 5 km el proxy casi mide "qué tan céntrica es la estación", así que parte de la correlación es cercanía al centro y no calles. Se conserva como una característica de **sitio** útil porque, a diferencia de la identidad de la estación (una indicadora por sitio que vale cero para una estación nunca vista), el proxy tiene un valor para cualquier punto del mapa, incluida una estación oculta en la validación; se propone 3 km como radio.

## 3. Resumen de las características

Rangos de las características de E6 en las 7,537 filas (ninguna tiene faltantes):

In [ ]:
eda.feature_summary(rows).round(2)

Las escalas son muy distintas: seno y coseno están en [−1, 1] y la ventilación va de 144 a 7,144 m²/s (la acumulada de 6 h, a 23,550). Eso importa para los métodos sensibles a la escala (PCA, regresión regularizada) y no importa para los árboles.

## 4. Escalamiento y transformaciones

### 4.1 Objetivo en escala logarítmica (E6)

E6 aprende `log(max(NOₓ − fondo, 1 ppb))`, con fondo de 5 ppb. Efecto de cada transformación posible, en asimetría y en cuánto de la varianza total concentran el 5 % de los valores más altos:

In [ ]:
eda.transform_effect(rows["nox"]).round(3)

En ppb la asimetría es 1.54 y el 5 % más alto de los valores aporta **44 %** de la varianza: un ajuste por mínimos cuadrados o un árbol que se parte para reducir el error cuadrático sigue casi solo esas horas de estancamiento. Con el logaritmo de NOₓ − fondo la asimetría pasa de 1.54 a −0.86 (menor en valor absoluto) y el 5 % más alto aporta **14 %**. El logaritmo hace además que un cambio relativo de emisión sea un **desplazamiento constante** en cualquier clima, que es la forma en que E6 lo mide (cociente de medias geométricas). El precio: los valores bajos se estiran (el log está sobre un NOₓ entero en ppb, así que hay escalones) y se necesita un piso, porque NOₓ − 5 puede ser cero o negativo:

In [ ]:
print(f"Filas en el piso de 1 ppb: {eda.floor_share(rows['nox']):.1%}")
eda.plot_transform(rows["nox"])

Solo el 1.9 % de las filas toca el piso de 1 ppb (columna de ceros del histograma), así que el piso casi no altera los resultados. La asimetría pasa de positiva a negativa (−0.86): se sobrecorrige un poco, con una cola izquierda corta. Para el modelo basta: el bosque no supone normalidad. Al invertir el logaritmo se corrige con el factor de *smearing* de Duan (en el modelo), porque la media del log no es el log de la media.

**Estandarización (z-score).** No se aplica al entrenar E6: los árboles dividen por umbrales y su resultado no cambia con la escala. Solo se estandariza donde la escala importaría: el PCA de la sección 5.3. Las inversiones bayesianas (E1–E4) trabajan en unidades físicas (ppb, t/año) con priors que ya tienen escala, y la PINN usa la ecuación adimensional de la siguiente sección.

### 4.2 Ecuación adimensional de la PINN

La PINN aprende **una sola función** en vez de una por hora. Con un viento uniforme por hora la ecuación de advección–difusión con decaimiento se ve igual, en el marco del viento, para todo el mundo salvo por dos números. Con longitudes en unidades de L = 10 km:

In [ ]:
eda.nondimensional_table()

Con eso, la columna que deja una fuente en un receptor depende solo de la posición en el marco del viento (A a lo largo, C a través) y de **γ = L / (|u| τ)** (cuántas veces cabe el decaimiento en el recorrido), con ε = κ/L fija. Es el cambio de variables que convierte 366 horas con 366 vientos y capas límite distintos en un mismo problema, que se aprende una vez y se reutiliza para toda hora y toda fuente: una sola red sirve para todo el periodo. Familia de soluciones exactas para varios γ (la red se valida contra ellas):

In [ ]:
eda.plot_green_function()

Cuanto mayor es γ (viento débil o vida corta del NOₓ), más rápido decae la columna río abajo. ¿Qué γ producen los vientos de los datos? Con vidas del NOₓ de 4, 8 y 12 horas (τ):

In [ ]:
eda.gamma_coverage(hours["speed"]).round(2)

La red se entrena para γ entre 0.03 y 0.75. Con τ = 8 h el 95 % de las horas queda dentro; con 12 h, el 98 %; con 4 h (vida corta) solo 73 %, porque con viento débil γ se pasa de 0.75. Fuera del rango la tabla de la red se **recorta al borde** (se usa el γ más cercano), así que la exactitud baja justo en las horas más estancadas y con τ cortos: es una limitación conocida que explica por qué τ es un parámetro a ajustar.

## 5. Selección y extracción de características

### 5.1 Correlación entre características

Spearman entre las nueve características de E6:

In [ ]:
corr = eda.correlation_matrix(rows, eda.FEATURES)
eda.plot_correlation(corr)

In [ ]:
eda.collinear_pairs(corr).round(3)

Solo dos pares superan 0.9:

- **`doy_sin` y `doy_cos` (1.00).** En 61 días el día del año recorre apenas una sexta parte del círculo (60°), y en ese arco seno y coseno son función monótona uno del otro: son **la misma variable, un contador de días**, no una estacionalidad. Es el hallazgo más importante de la sección: con dos meses no hay estacionalidad que aprender, y un árbol que parta por `doy` solo memoriza qué días tuvieron cierto clima no explicado. Y en el periodo de prueba (feb–may 2024) esos valores quedan **fuera del rango de ajuste**, donde un árbol no extrapola (asigna la hoja del extremo).
- **`ventilation_3h` y `ventilation_6h` (0.985).** La suma de 6 horas contiene a la de 3: son casi redundantes.

### 5.2 Varianza

In [ ]:
eda.variance_table(rows, eda.FEATURES).round(3)

Ninguna característica es constante: la que menos variedad tiene (`hour_*`) toma seis valores (una hora cada vez, 17 % de las filas cada una); las demás tienen 61 a 366 valores distintos. El coeficiente de variación de `wind_cos` (3.7) refleja que su media está cerca de cero, no que sea ruido. **Ninguna se descarta por varianza.**

### 5.3 Extracción: PCA donde tiene sentido

El PCA tiene sentido en el bloque de variables **continuas y colineales de clima** (las tres ventilaciones, la velocidad y la capa límite, estandarizadas). No lo tiene en pares de seno y coseno (ya son una codificación de dos dimensiones de una variable), ni en indicadoras de tipo de día o de estación (categóricas).

In [ ]:
block = [*eda.VENTILATION, "speed", "pbl"]
explained, loadings = eda.pca_table(rows, block)
explained.round(3)

In [ ]:
loadings.round(2)

Las cinco variables de clima ocupan **tres dimensiones**: tres componentes explican 98.9 % de la varianza. El primero (68 %) es "ventilación general" (los cuatro pesos de la ventilación y la velocidad van juntos, 0.44–0.51); el segundo (22 %) es la capa límite frente a la velocidad (+0.88 y −0.45); el tercero separa la ventilación de la hora de las acumuladas. Los componentes 4 y 5 (< 1 %) son esencialmente la diferencia entre las acumuladas de 3 y 6 horas. Segundo uso del PCA, sobre el NOₓ de las estaciones (log, horas con todas las estaciones presentes):

In [ ]:
station_explained = eda.station_pca(rows)
print(f"{station_explained.attrs['stations']} estaciones, {station_explained.attrs['hours']} horas completas")
station_explained.head(6).round(3)

In [ ]:
eda.plot_pca(explained, station_explained, loadings)

El primer componente de las estaciones explica **69 %** de la varianza: la ciudad se mueve en bloque porque comparte clima, y por eso el NOₓ de cada hora no basta para ver la emisión. Los demás componentes explican poco cada uno (10, 6, 3 %): los patrones espaciales que quedan, los que informan *dónde* se emite, son pocos y pequeños. Solo hay 154 horas con las 18 estaciones más completas a la vez, y por eso el PCA se limita a ese subconjunto.

**Decisión sobre el PCA:** se usa como *diagnóstico* y no como transformación del modelo. Reemplazar la ventilación por componentes ahorraría dos columnas, pero se perdería la interpretación física (m²/s) y los árboles no sufren por la colinealidad.

### 5.4 Relevancia para el objetivo

Para cada característica, la correlación absoluta (monótona) y la información mutua (cualquier relación, en nats) con el objetivo de E6, vistas de una en una:

In [ ]:
eda.feature_relevance(rows).round(3)

La ventilación y sus acumuladas (|ρ| = 0.32–0.34) y la hora (0.18–0.28) son las más relevantes por correlación, y la información mutua pone también a la ventilación arriba. Pero la información mutua más alta es la de **`doy_sin` y `doy_cos` (0.29 y 0.28)**, con correlación de apenas 0.16: es la señal de que el día del año funciona como identificador de día (un día tuvo un clima y emisión que no capturan las demás variables). Una característica así *parece* informativa en el ajuste y no se traslada a otro periodo. La dirección del viento aporta (0.14–0.16 nats) aunque correlaciona poco con el NOₓ: es una relación que no es monótona (depende de qué lado esté la fuente respecto de cada estación).

### 5.5 Información por tipo de observación

No solo hay que elegir *variables*: también *tipos de observación*. Con la inversión bayesiana lineal (línea base, `inversion_plume`), la información que cada fuente de datos le da a las incógnitas (4 fuentes puntuales y un factor de emisión vial por zona y tipo de día: 31 en total) no depende de los valores observados, solo de dónde y cuándo se midió. Se calcula la reducción de la varianza a posteriori con solo estaciones, solo TEMPO y ambas, y se resume en grados de libertad para la señal (cuántas de las 31 incógnitas se determinan):

In [ ]:
reduction, dofs = eda.observation_information(inputs)
dofs.round(2)

In [ ]:
eda.plot_information(dofs)

- Las **estaciones** determinan 26.2 de las 31 incógnitas: 8.5 de 9 factores de día hábil, 7.2 y 7.4 de los de sábado y domingo (tienen menos horas) y apenas 3.0 de las 4 fuentes puntuales (están lejos de casi todas las estaciones).
- **TEMPO** determina 31.0: casi todo, con 78,227 píxeles. **Las dos juntas dan lo mismo que TEMPO solo.**
- Interpretación prudente: es información **formal**, bajo errores independientes por píxel; los errores reales de un satélite están correlacionados y tienen sesgo, así que 31.0 es un máximo optimista. Aun así muestra que TEMPO es el que **ve las fuentes puntuales** (desviación a posteriori de 33–39 con TEMPO frente a 165–696 con solo estaciones, en las unidades del prior; ver abajo) y que las estaciones son el ancla de la señal en superficie. Cuánto pesa TEMPO frente a las estaciones se trata como hiperparámetro en el ajuste fino, no aquí.

In [ ]:
reduction[["label", "prior_sd", "sd_stations", "sd_tempo", "sd_both"]].head(4).round(1)

## 6. Conclusiones de la preparación de datos (CRISP-ML(Q))

Fase de **preparación de datos**, en las cinco tareas de CRISP-ML(Q) más su aseguramiento de calidad:

| Tarea | Qué se hizo | Evidencia | Riesgo o control de calidad |
|---|---|---|---|
| **Seleccionar** | Noviembre–diciembre de 2023, tardes, estaciones con NOₓ (24 con coordenadas) | Avance 1 (4.1–4.3, 9) | El periodo de prueba (feb–may 2024) no se carga en este cuaderno; `nox-ml tune` rechaza configuraciones que lo incluyan |
| **Limpiar** | Viento 2024 de mph a m/s; sensores atascados enmascarados; filtros de TEMPO; sin imputar NOₓ | Avance 1 (5.2–5.4) | Efecto nulo del enmascarado en estos periodos; 19 filas de 2025 sin corregir (fuera de periodo) |
| **Construir** | Ventilación y sus acumuladas, hora y día cíclicos, dirección en seno/coseno, tipo de día, proxy vial de OSM | Secciones 2 y 3 | `doy_*` es un contador de días; `ventilation_3h` y `_6h` son casi redundantes |
| **Integrar** | Tres fuentes con una convención de tiempo (sin zona, hora de inicio) y una malla UTM; un `Inputs` con estaciones, viento, capa límite y píxeles | `load_inputs`, `hourly_table`, `feature_rows` | Estaciones con datos pero fuera del dominio (TAH) o sin NOₓ (4) no entran |
| **Formatear** | Objetivo logarítmico con piso para E6; ecuación adimensional para la PINN; sin escalar para árboles | Sección 4 | Piso de 1 ppb toca 1.9 % de las filas; la PINN recorta γ fuera de 0.03–0.75 (27 % de las horas con τ = 4 h) |

**Decisiones y hallazgos de este avance**

1. Las **características de E6 están justificadas físicamente** (ventilación, memoria de 3 y 6 h, ciclos, dirección, tipo de día) y se calculan con el mismo código que usa el modelo.
2. **La ventilación aporta por su forma física, no por su correlación:** la capa límite sola correlaciona más con el NOₓ (−0.41 frente a −0.34).
3. **`doy_sin/cos` no son estacionalidad en dos meses** (correlación de 1.00 entre sí, información mutua alta con baja correlación) y quedarán fuera del rango en el periodo de prueba. **Recomendación:** evaluar E6 sin ellas con la validación con estación oculta (ablación) antes del avance 5; no se cambia el modelo aquí porque la decisión debe salir de esa comparación y no de este análisis.
4. **`ventilation_3h` y `_6h` casi redundan** (0.985). Se conservan por ahora (los árboles toleran la colinealidad, E6 ya está validado), pero son candidatas a una ablación.
5. **El proxy vial de OSM ordena bien a las estaciones** (Spearman 0.71 a 3 km, 24 estaciones) y puede ser una característica de sitio que sí exista para una estación oculta; falta probar si mejora el error con estación oculta antes de adoptarla.
6. **El logaritmo del objetivo es la transformación correcta para E6:** reduce a 14 % la varianza que mandan los valores más altos (de 44 %) y hace que el cambio de emisión sea un desplazamiento constante.
7. **TEMPO y estaciones informan cosas distintas:** TEMPO determina las fuentes puntuales y las estaciones el nivel en superficie; el peso relativo de TEMPO es un hiperparámetro a ajustar con validación y no una decisión cerrada.

**Riesgos que quedan:** dos meses de datos (sin estacionalidad que validar), 41/9/11 días por tipo de día, información formal de TEMPO optimista, y la sensibilidad del periodo de prueba a la capa límite de MERRA-2 (Avance 1, sección 4.4).

**Aseguramiento de calidad.** Todos los números de este cuaderno salen de funciones de `nox_ml.validation.report_eda` con pruebas unitarias sobre datos sintéticos de respuesta conocida (`ml/tests/test_report_eda.py`); el cuaderno se ejecuta completo sobre los datos reales en la validación completa (`pytest -m slow`).